In [ ]:
# ============================================
# DÉFI QUOTIDIEN : CLASSIFICATION DE TEXTE IMDB
# CLASSIFICATION BINAIRE AVEC CNN 1D
# ============================================

# ============================================
# PARTIE 1 : IMPORTATION DES BIBLIOTHÈQUES
# ============================================
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
np.random.seed(42)
tf.random.set_seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("="*50)
print("DÉFI QUOTIDIEN : CLASSIFICATION IMDB")
print("CLASSIFICATION BINAIRE AVEC CNN")
print("="*50)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

# ============================================
# PARTIE 2 : CHARGEMENT ET EXPLORATION DES DONNÉES
# ============================================
print("\n" + "="*50)
print("CHARGEMENT DES DONNÉES IMDB")
print("="*50)

# Paramètres
MAX_FEATURES = 10000  # Nombre de mots les plus fréquents à conserver
MAX_LENGTH = 500      # Longueur maximale des séquences
EMBEDDING_DIM = 100   # Dimension de l'embedding

# Chargement du dataset
(X_train_original, y_train_original), (X_test_original, y_test_original) = imdb.load_data(num_words=MAX_FEATURES)

print(f"Taille de l'ensemble d'entraînement: {len(X_train_original)}")
print(f"Taille de l'ensemble de test: {len(X_test_original)}")
print(f"Nombre de classes: {len(np.unique(y_train_original))}")
print(f"Distribution des classes (entraînement):")
print(f"  - Positives: {sum(y_train_original == 1)}")
print(f"  - Négatives: {sum(y_train_original == 0)}")

# Décodage des séquences pour visualisation
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

def decode_review(sequence_words, reverse_word_index):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in sequence_words])

print("\nExemple de critique positive:")
print(decode_review(X_train_original[0], reverse_word_index))
print(f"Étiquette: {'Positive' if y_train_original[0] == 1 else 'Négative'}")

print("\nExemple de critique négative:")
print(decode_review(X_train_original[1], reverse_word_index))
print(f"Étiquette: {'Positive' if y_train_original[1] == 1 else 'Négative'}")

# Distribution des longueurs
review_lengths = [len(review) for review in X_train_original]
plt.figure(figsize=(12, 5))
plt.hist(review_lengths, bins=50, color='skyblue', edgecolor='black')
plt.axvline(MAX_LENGTH, color='red', linestyle='--', label=f'Longueur max: {MAX_LENGTH}')
plt.title('Distribution des longueurs des critiques')
plt.xlabel('Longueur de la critique')
plt.ylabel('Nombre de critiques')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nStatistiques des longueurs:")
print(f"  - Longueur moyenne: {np.mean(review_lengths):.2f}")
print(f"  - Longueur médiane: {np.median(review_lengths):.2f}")
print(f"  - Longueur max: {max(review_lengths)}")
print(f"  - Longueur min: {min(review_lengths)}")

# ============================================
# PARTIE 3 : PRÉTRAITEMENT DES DONNÉES
# ============================================
print("\n" + "="*50)
print("PRÉTRAITEMENT DES DONNÉES")
print("="*50)

# 3.1 : Padding des séquences (approche alternative au one-hot encoding)
print("Option 1: Padding des séquences (pour CNN 1D)")
X_train_pad = tf.keras.preprocessing.sequence.pad_sequences(X_train_original, maxlen=MAX_LENGTH)
X_test_pad = tf.keras.preprocessing.sequence.pad_sequences(X_test_original, maxlen=MAX_LENGTH)
print(f"Shape de X_train_pad: {X_train_pad.shape}")
print(f"Shape de X_test_pad: {X_test_pad.shape}")

# 3.2 : One-hot encoding (pour réseau dense)
print("\nOption 2: One-hot encoding (pour réseau dense)")
def vectorize_sequences(sequences, dimension=MAX_FEATURES):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1
    return results

X_train_onehot = vectorize_sequences(X_train_original, MAX_FEATURES)
X_test_onehot = vectorize_sequences(X_test_original, MAX_FEATURES)
print(f"Shape de X_train_onehot: {X_train_onehot.shape}")
print(f"Shape de X_test_onehot: {X_test_onehot.shape}")
print(f"Mémoire utilisée: {X_train_onehot.nbytes / 1024**2:.2f} MB")

# 3.3 : Division en train/validation
from sklearn.model_selection import train_test_split

# Pour les séquences pad
X_train, X_val, y_train, y_val = train_test_split(
    X_train_pad, y_train_original, test_size=0.15, random_state=42
)
print(f"\nDivision des données (séquences pad):")
print(f"  - Train: {len(X_train)}")
print(f"  - Validation: {len(X_val)}")
print(f"  - Test: {len(X_test_pad)}")

# Pour le one-hot
X_train_oh, X_val_oh, y_train_oh, y_val_oh = train_test_split(
    X_train_onehot, y_train_original, test_size=0.15, random_state=42
)
print(f"\nDivision des données (one-hot):")
print(f"  - Train: {len(X_train_oh)}")
print(f"  - Validation: {len(X_val_oh)}")
print(f"  - Test: {len(X_test_onehot)}")

# ============================================
# PARTIE 4 : MODÈLE CNN 1D (RECOMMANDÉ POUR LE TEXTE)
# ============================================
print("\n" + "="*50)
print("CONSTRUCTION DU MODÈLE CNN 1D")
print("="*50)

def create_text_cnn_model(vocab_size=MAX_FEATURES, embedding_dim=EMBEDDING_DIM, max_length=MAX_LENGTH):
    """
    Crée un modèle CNN 1D pour la classification de texte
    """
    model = models.Sequential([
        # Couche d'embedding
        layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
        
        # Blocs de convolution 1D
        layers.Conv1D(128, 5, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(5),
        
        layers.Conv1D(128, 5, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(5),
        
        layers.Conv1D(128, 5, activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalMaxPooling1D(),
        
        # Partie dense
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

# Création du modèle CNN
cnn_model = create_text_cnn_model()
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Architecture du modèle CNN 1D:")
cnn_model.summary()

# ============================================
# PARTIE 5 : MODÈLE DENSE (RÉSEAU À PROPAGATION AVANT)
# ============================================
print("\n" + "="*50)
print("CONSTRUCTION DU MODÈLE DENSE (PROPAGATION AVANT)")
print("="*50)

def create_dense_model(input_dim=MAX_FEATURES):
    """
    Crée un modèle dense pour la classification de texte avec one-hot encoding
    """
    model = models.Sequential([
        layers.Dense(512, activation='relu', input_shape=(input_dim,)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(1, activation='sigmoid')
    ])
    return model

# Création du modèle dense
dense_model = create_dense_model()
dense_model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Architecture du modèle Dense:")
dense_model.summary()

# ============================================
# PARTIE 6 : CALLBACKS POUR L'ENTRAÎNEMENT
# ============================================
print("\n" + "="*50)
print("CONFIGURATION DES CALLBACKS")
print("="*50)

callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_imdb_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("Callbacks configurés:")
print("  - EarlyStopping (patience: 5)")
print("  - ReduceLROnPlateau (patience: 3)")
print("  - ModelCheckpoint (sauvegarde du meilleur)")

# ============================================
# PARTIE 7 : ENTRAÎNEMENT DU MODÈLE CNN
# ============================================
print("\n" + "="*50)
print("ENTRAÎNEMENT DU MODÈLE CNN 1D")
print("="*50)

EPOCHS = 20
BATCH_SIZE = 512

cnn_history = cnn_model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks_list,
    verbose=1
)

# ============================================
# PARTIE 8 : ENTRAÎNEMENT DU MODÈLE DENSE
# ============================================
print("\n" + "="*50)
print("ENTRAÎNEMENT DU MODÈLE DENSE")
print("="*50)

dense_history = dense_model.fit(
    X_train_oh, y_train_oh,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val_oh, y_val_oh),
    callbacks=callbacks_list,
    verbose=1
)

# ============================================
# PARTIE 9 : VISUALISATION DES PERFORMANCES
# ============================================
print("\n" + "="*50)
print("VISUALISATION DES PERFORMANCES")
print("="*50)

def plot_training_history(history, model_name, metrics=['accuracy', 'loss']):
    """
    Visualise les métriques d'entraînement
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Précision
    axes[0].plot(history.history['accuracy'], label='Entraînement', color='blue')
    axes[0].plot(history.history['val_accuracy'], label='Validation', color='red')
    axes[0].set_title(f'{model_name} - Précision')
    axes[0].set_xlabel('Époque')
    axes[0].set_ylabel('Précision')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Perte
    axes[1].plot(history.history['loss'], label='Entraînement', color='blue')
    axes[1].plot(history.history['val_loss'], label='Validation', color='red')
    axes[1].set_title(f'{model_name} - Perte')
    axes[1].set_xlabel('Époque')
    axes[1].set_ylabel('Perte')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Analyse du surapprentissage
    train_acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]
    train_loss = history.history['loss'][-1]
    val_loss = history.history['val_loss'][-1]
    
    print(f"\n{model_name} - Métriques finales:")
    print(f"  Précision - Entraînement: {train_acc:.4f}")
    print(f"  Précision - Validation: {val_acc:.4f}")
    print(f"  Écart de précision: {(train_acc - val_acc)*100:.2f}%")
    
    if train_acc - val_acc > 0.05:
        print("  ⚠️ Surapprentissage détecté!")
    else:
        print("  ✅ Pas de surapprentissage significatif.")
    
    return train_acc, val_acc

# Visualisation pour les deux modèles
cnn_train_acc, cnn_val_acc = plot_training_history(cnn_history, 'CNN 1D')
dense_train_acc, dense_val_acc = plot_training_history(dense_history, 'Dense (One-Hot)')

# ============================================
# PARTIE 10 : ÉVALUATION SUR L'ENSEMBLE DE TEST
# ============================================
print("\n" + "="*50)
print("ÉVALUATION SUR L'ENSEMBLE DE TEST")
print("="*50)

# Pour le CNN 1D
print("\nModèle CNN 1D:")
test_loss_cnn, test_acc_cnn = cnn_model.evaluate(X_test_pad, y_test_original, verbose=0)
print(f"  Précision sur le test: {test_acc_cnn:.4f}")
print(f"  Perte sur le test: {test_loss_cnn:.4f}")

# Pour le modèle Dense
print("\nModèle Dense (One-Hot):")
test_loss_dense, test_acc_dense = dense_model.evaluate(X_test_onehot, y_test_original, verbose=0)
print(f"  Précision sur le test: {test_acc_dense:.4f}")
print(f"  Perte sur le test: {test_loss_dense:.4f}")

# ============================================
# PARTIE 11 : MATRICE DE CONFUSION ET RAPPORT
# ============================================
print("\n" + "="*50)
print("MATRICE DE CONFUSION")
print("="*50)

def evaluate_predictions(model, X_test, y_test, model_name):
    """
    Génère la matrice de confusion et le rapport de classification
    """
    # Prédictions
    predictions = model.predict(X_test)
    predicted_classes = (predictions > 0.5).astype(int).flatten()
    
    # Matrice de confusion
    cm = confusion_matrix(y_test, predicted_classes)
    
    # Affichage
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Matrice de confusion
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Négatives', 'Positives'],
                yticklabels=['Négatives', 'Positives'],
                ax=axes[0])
    axes[0].set_title(f'{model_name} - Matrice de Confusion')
    axes[0].set_xlabel('Prédictions')
    axes[0].set_ylabel('Vraies Valeurs')
    
    # Rapport de classification
    report = classification_report(y_test, predicted_classes, 
                                  target_names=['Négatives', 'Positives'], 
                                  output_dict=True)
    metrics_df = pd.DataFrame(report).T.iloc[:2, :3]
    metrics_df.plot(kind='bar', ax=axes[1])
    axes[1].set_title(f'{model_name} - Précision et Rappel')
    axes[1].set_xlabel('Classe')
    axes[1].set_ylabel('Score')
    axes[1].set_ylim([0, 1])
    axes[1].legend(loc='lower right')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n{model_name} - Rapport de classification:")
    print(classification_report(y_test, predicted_classes, target_names=['Négatives', 'Positives']))
    
    return predicted_classes

# Évaluation des deux modèles
print("\nÉvaluation du modèle CNN 1D:")
cnn_preds = evaluate_predictions(cnn_model, X_test_pad, y_test_original, 'CNN 1D')

print("\nÉvaluation du modèle Dense:")
dense_preds = evaluate_predictions(dense_model, X_test_onehot, y_test_original, 'Dense')

# ============================================
# PARTIE 12 : PRÉDICTION SUR DES TEXTES PERSONNALISÉS
# ============================================
print("\n" + "="*50)
print("PRÉDICTION SUR DES TEXTES PERSONNALISÉS")
print("="*50)

def preprocess_text(text, max_len=MAX_LENGTH):
    """
    Prétraite un texte pour la prédiction
    """
    # Tokenisation simple
    words = text.lower().split()
    
    # Encodage
    word_index = imdb.get_word_index()
    encoded = []
    for word in words:
        if word in word_index:
            # +3 pour les indices IMDB
            encoded.append(word_index[word] + 3)
        else:
            encoded.append(2)  # Token inconnu
    
    # Padding
    padded = tf.keras.preprocessing.sequence.pad_sequences([encoded], maxlen=max_len)
    return padded

def predict_review(model, text, model_type='cnn'):
    """
    Fait une prédiction sur une critique personnalisée
    """
    # Prétraitement
    if model_type == 'cnn':
        processed = preprocess_text(text)
    else:
        # Pour le modèle dense, on utilise le one-hot
        words = text.lower().split()
        word_index = imdb.get_word_index()
        encoded = []
        for word in words:
            if word in word_index:
                encoded.append(word_index[word] + 3)
            else:
                encoded.append(2)
        # One-hot encoding
        processed = vectorize_sequences([encoded], MAX_FEATURES)
    
    # Prédiction
    prediction = model.predict(processed)
    confidence = prediction[0][0]
    sentiment = 'Positive' if confidence > 0.5 else 'Négative'
    confidence_percent = confidence * 100 if confidence > 0.5 else (1 - confidence) * 100
    
    print(f"Critique: {text}")
    print(f"Sentiment: {sentiment} (Confiance: {confidence_percent:.2f}%)")
    print("-" * 40)
    
    return sentiment, confidence

# Test avec des exemples
print("\nTest avec des critiques personnalisées (CNN 1D):")
print("-" * 40)

test_reviews = [
    "This movie was absolutely fantastic! Great acting and wonderful story.",
    "Terrible film, waste of time. Bad acting and boring plot.",
    "Decent movie, not the best but enjoyable enough.",
    "Amazing cinematography and brilliant performances by all actors.",
    "I didn't like it at all, very disappointing and poorly made."
]

for review in test_reviews:
    predict_review(cnn_model, review, 'cnn')

# ============================================
# PARTIE 13 : COMPARAISON DES MODÈLES
# ============================================
print("\n" + "="*50)
print("COMPARAISON DES MODÈLES")
print("="*50)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Comparaison des précisions de validation
axes[0].plot(cnn_history.history['val_accuracy'], label='CNN 1D', color='blue')
axes[0].plot(dense_history.history['val_accuracy'], label='Dense (One-Hot)', color='red')
axes[0].set_title('Comparaison des précisions de validation')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Précision')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Comparaison des pertes de validation
axes[1].plot(cnn_history.history['val_loss'], label='CNN 1D', color='blue')
axes[1].plot(dense_history.history['val_loss'], label='Dense (One-Hot)', color='red')
axes[1].set_title('Comparaison des pertes de validation')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Perte')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Métriques comparatives
print("\nComparaison des performances:")
print("-" * 40)
print(f"Modèle CNN 1D:")
print(f"  - Précision validation: {cnn_val_acc:.4f}")
print(f"  - Précision test: {test_acc_cnn:.4f}")
print(f"  - Perte test: {test_loss_cnn:.4f}")
print(f"  - Paramètres: {cnn_model.count_params():,}")

print(f"\nModèle Dense (One-Hot):")
print(f"  - Précision validation: {dense_val_acc:.4f}")
print(f"  - Précision test: {test_acc_dense:.4f}")
print(f"  - Perte test: {test_loss_dense:.4f}")
print(f"  - Paramètres: {dense_model.count_params():,}")

# ============================================
# PARTIE 14 : SAUVEGARDE DES ARTEFACTS
# ============================================
print("\n" + "="*50)
print("SAUVEGARDE DES ARTEFACTS")
print("="*50)

# Sauvegarde des modèles
cnn_model.save('imdb_cnn_model.h5')
dense_model.save('imdb_dense_model.h5')
print("✅ Modèles sauvegardés:")
print("  - imdb_cnn_model.h5")
print("  - imdb_dense_model.h5")

# Sauvegarde de la configuration
import json
from datetime import datetime

config = {
    'models': {
        'cnn': {
            'architecture': 'CNN 1D avec Embedding',
            'max_features': MAX_FEATURES,
            'max_length': MAX_LENGTH,
            'embedding_dim': EMBEDDING_DIM,
            'parameters': cnn_model.count_params(),
            'test_accuracy': float(test_acc_cnn),
            'test_loss': float(test_loss_cnn)
        },
        'dense': {
            'architecture': 'Dense avec One-Hot Encoding',
            'max_features': MAX_FEATURES,
            'parameters': dense_model.count_params(),
            'test_accuracy': float(test_acc_dense),
            'test_loss': float(test_loss_dense)
        }
    },
    'training': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'optimizer': 'Adam/RMSprop',
        'loss': 'binary_crossentropy'
    },
    'timestamp': datetime.now().isoformat()
}

with open('imdb_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print("✅ Configuration sauvegardée: imdb_config.json")

# ============================================
# PARTIE 15 : RÉSUMÉ FINAL
# ============================================
print("\n" + "="*50)
print("RÉSUMÉ FINAL - DÉFI IMDB")
print("="*50)
print(f"""
✅ Classification de texte IMDB réussie!

Performances CNN 1D:
- Précision validation: {cnn_val_acc:.4f} ({cnn_val_acc*100:.2f}%)
- Précision test: {test_acc_cnn:.4f} ({test_acc_cnn*100:.2f}%)
- Perte test: {test_loss_cnn:.4f}
- Paramètres: {cnn_model.count_params():,}

Performances Dense (One-Hot):
- Précision validation: {dense_val_acc:.4f} ({dense_val_acc*100:.2f}%)
- Précision test: {test_acc_dense:.4f} ({test_acc_dense*100:.2f}%)
- Perte test: {test_loss_dense:.4f}
- Paramètres: {dense_model.count_params():,}

Architectures:
- CNN 1D: Embedding + 3 Conv1D + GlobalMaxPooling + 2 Dense
- Dense: 3 couches cachées avec Dropout et BatchNormalization

Compétences acquises:
✅ Prétraitement de texte pour réseaux neuronaux
✅ Architectures CNN et dense pour classification binaire
✅ Évaluation et détection du surapprentissage
✅ Visualisation des métriques d'entraînement
✅ Prédiction sur des textes personnalisés

Livrables:
1. imdb_cnn_model.h5 - Modèle CNN 1D
2. imdb_dense_model.h5 - Modèle Dense
3. imdb_config.json - Configuration
4. Visualisations et métriques
""")

print("="*50)
print("🎉 DÉFI QUOTIDIEN IMDB COMPLET! FÉLICITATIONS!")
print("="*50)